# 06 - Reproducing the Marchwicka--Politarczyk computation

This notebook reconstructs the computer-assisted lower-bound calculation for the knot in Theorem 1.1 of Maria Marchwicka and Wojciech Politarczyk, *On the slice genus of generalized algebraic knots*. It follows the paper's notation and proof structure while using the modern, exact, coverage-aware `gaknot` API.

The published conclusion is

`g_4^top(K) = g_4(K) = 2`.

The software calculation reproduced here establishes the difficult lower bound `g_4^top(K) >= 2`: for both the 83- and 103-primary parts, every nonzero projective isotropic class has a scalar multiple that violates the genus-one Casson--Gordon inequality. The paper's geometric construction in Lemma 4.1 supplies the upper bound; this notebook describes that input but does not claim to construct the surface computationally.

An optional exhaustive text log records every lookup-table value, isotropic line, multiplier, character vector, component contribution, signature, nullity, and inequality comparison actually used by the optimized search.

## 1. Setup

In [ ]:
from dataclasses import asdict
from pathlib import Path
import sys

repository_root = Path.cwd()
if repository_root.name == "notebooks":
    repository_root = repository_root.parent
source_directory = repository_root / "src"
if str(source_directory) not in sys.path:
    sys.path.insert(0, str(source_directory))

from sage.all import QQ
from gaknot import (
    GeneralizedAlgebraicKnot,
    PrimeDiagonalLinkingForm,
)

## 2. The eight-summand knot from Theorem 1.1

With the convention that a cable sequence is written from the innermost knot outward, the paper's knot is

`T(2,17;2,83) # -T(2,11;2,83) # T(2,83) # -T(2,13;2,83)`

`# T(2,11;2,103) # -T(2,103) # T(2,13;2,103) # -T(2,17;2,103)`.

The first four summands form the 83-primary block and the last four form the 103-primary block. Keeping this order visible is essential because later character vectors use one coordinate per displayed summand.

In [ ]:
paper_knot = GeneralizedAlgebraicKnot([
    # The 83-primary block.
    (1, [(2, 17), (2, 83)]),
    (-1, [(2, 11), (2, 83)]),
    (1, [(2, 83)]),
    (-1, [(2, 13), (2, 83)]),
    # The 103-primary block.
    (1, [(2, 11), (2, 103)]),
    (-1, [(2, 103)]),
    (1, [(2, 13), (2, 103)]),
    (-1, [(2, 17), (2, 103)]),
])

print("K =", paper_knot)
print("number of connected-sum summands:", len(paper_knot))
for index, (sign, cable_sequence) in enumerate(paper_knot.description):
    print(index, "sign=", sign, "sequence=", cable_sequence)

The paper proves that this knot is algebraically slice. Vanishing of the Levine--Tristram signature is a necessary consistency check, but it is not by itself a proof of algebraic sliceness. The following computation verifies that the implemented signature function cancels identically across the eight summands.

In [ ]:
ordinary_signature = paper_knot.signature()

print("number of nonzero signature jumps:",
      len(ordinary_signature.jumps_counter))
print("signature vanishes everywhere:",
      ordinary_signature.is_zero_everywhere())
print("classical signature at -1:", ordinary_signature(QQ(1) / 2))

## 3. The distinguished double-cover linking form

Every supported `(2,q)` cable contributes a distinguished `Z/qZ` coordinate. A positive summand has self-pairing `-1/q`, and reversing its concordance orientation changes the sign. Consequently, the double-cover homology is `(Z/83Z)^4 + (Z/103Z)^4`, with a split diagonal form on each primary part.

In [ ]:
linking_form = PrimeDiagonalLinkingForm.from_knot(paper_knot)

print(linking_form)
print("orders:", linking_form.orders)
print("diagonal numerators:", linking_form.coefficients)
print("primary primes:", linking_form.primary_primes)
for prime in linking_form.primary_primes:
    print(f"indices of the {prime}-primary part:",
          linking_form.primary_indices(prime))

The coordinate order used by the implementation follows the displayed knot. The paper permutes the third and fourth coordinates in its formulas for `Q_q` and `Sigma_q`, and an overall sign may appear when switching between the linking form and its numerator. Neither operation changes which vectors are isotropic. The explicit `orders`, `coefficients`, and component indices above are the authoritative convention for all vectors below.

## 4. Isotropic vectors and metabolizers

A metabolizer is a half-dimensional totally isotropic subspace. The following four generators give one concrete metabolizer of the complete linking form: two generators span a metabolizer of the 83-primary block and two span one for the 103-primary block.

This example is useful for checking the definition, but it is **not** how Algorithm 1 proves the obstruction. The proof must rule out the metabolizer that would arise from an unknown genus-one surface. Instead of guessing that subspace, it checks every projective isotropic line: any nonzero primary metabolizer necessarily contains one of them.

In [ ]:
example_metabolizer_generators = [
    (1, 1, 0, 0, 0, 0, 0, 0),
    (0, 0, 1, 1, 0, 0, 0, 0),
    (0, 0, 0, 0, 1, 1, 0, 0),
    (0, 0, 0, 0, 0, 0, 1, 1),
]

for generator in example_metabolizer_generators:
    print(
        generator,
        "self-pairing=", linking_form.pairing(generator, generator),
        "isotropic=", linking_form.is_isotropic(generator),
    )
print("the four generators span a metabolizer:",
      linking_form.is_metabolizer(example_metabolizer_generators))

## 5. Casson--Gordon data for one isotropic line

To make the inner loop transparent before running the complete search, take the 83-primary vector `x=(1,1,0,0)`. In the full eight-coordinate form it is isotropic. A homology element `k*x` determines character parameters by multiplying each coordinate by the corresponding diagonal linking-form numerator.

For genus one and classical signature zero, Gilmer's allowed bound is `eta+5`. Algorithm 1 varies `k` until the absolute Casson--Gordon signature exceeds this bound. Conjugation identifies `k` with `83-k`, so the optimized implementation only needs `1 <= k <= 41`.

In [ ]:
isotropic_83 = (1, 1, 0, 0, 0, 0, 0, 0)
assert linking_form.is_isotropic(isotropic_83)

first_hand_witness = None
for multiple in range(1, 83 // 2 + 1):
    parameters = tuple(
        (coefficient * multiple * coordinate) % order
        for coefficient, coordinate, order in zip(
            linking_form.coefficients,
            isotropic_83,
            linking_form.orders,
        )
    )
    invariant = paper_knot.casson_gordon(parameters)
    bound = invariant.eta + 5
    if abs(invariant.sigma) > bound:
        first_hand_witness = (
            multiple, parameters, invariant.sigma, invariant.eta, bound
        )
        break

print("first violating multiple for this line:", first_hand_witness)

The deliberately expanded calculation above uses the public Casson--Gordon API. The full search uses precomputed integral tables for `q*sigma` instead, avoiding repeated evaluation of the same component formula while preserving exact arithmetic.

## 6. Configure the optional exhaustive text log

Set `WRITE_FULL_LOG=True` to create `computation_logs/marchwicka_politarczyk_gilmer_search.txt`. The directory is ignored by Git because a detailed theorem-sized audit can be large. With the flag left `False`, the complete mathematical search still runs but performs no file I/O.

The default file mode is `w`, which replaces an earlier log at the same path. Use `log_mode="a"` when deliberately collecting several runs in one file. The destination directory must exist; this cell creates it only when logging is requested.

In [ ]:
WRITE_FULL_LOG = False
FULL_LOG_PATH = (
    repository_root
    / "computation_logs"
    / "marchwicka_politarczyk_gilmer_search.txt"
)

if WRITE_FULL_LOG:
    FULL_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    print("The exhaustive audit will be written to:", FULL_LOG_PATH)
else:
    print("Exhaustive file logging is disabled.")
    print("Set WRITE_FULL_LOG=True and rerun from this cell to enable it.")

## 7. Reproduce Lemma 3.1 and the genus-one obstruction

The search below is the notebook's main computation. For each primary block it enumerates the unique canonical representative of every nonzero isotropic projective line. It then searches scalar multiples until it finds a strict violation of

`|sigma(K,chi_x) + sigma_K| <= eta(K,chi_x) + 4*g + 1`

with `g=1`. Certification requires success on **every** isotropic line in an eligible primary block.

In [ ]:
if WRITE_FULL_LOG:
    # Add notebook-level data that precede Algorithm 1. The core search then
    # appends its exhaustive, deterministic audit to the same text file.
    with FULL_LOG_PATH.open("w", encoding="utf-8", newline="\n") as log_file:
        log_file.write("MARCHWICKA--POLITARCZYK NOTEBOOK PREAMBLE\n")
        log_file.write(f"ordinary_signature_is_zero={ordinary_signature.is_zero_everywhere()}\n")
        log_file.write("EXAMPLE_METABOLIZER_BEGIN\n")
        for generator in example_metabolizer_generators:
            log_file.write(
                f"generator={generator} "
                f"self_pairing={linking_form.pairing(generator, generator)}\n"
            )
        log_file.write(
            "verified_metabolizer="
            f"{linking_form.is_metabolizer(example_metabolizer_generators)}\n"
        )
        log_file.write("EXAMPLE_METABOLIZER_END\n\n")

obstruction = paper_knot.gilmer_genus_obstruction(
    1,
    log_path=FULL_LOG_PATH if WRITE_FULL_LOG else None,
    log_mode="a" if WRITE_FULL_LOG else "w",
)

print("tested genus:", obstruction.tested_genus)
print("classical signature:", obstruction.classical_signature)
print("certified:", obstruction.certified)
print("lower bound:", obstruction.lower_bound)
print("successful primes:", obstruction.successful_primes)

The expected lower bound is two. Both primary parts independently certify it, which is stronger than the minimum needed for the final contradiction.

## 8. Audit the complete finite searches

A four-dimensional vector space over `F_q` has `q^3+q^2+q+1` projective lines. The split quadric defined by the primary linking form contains `(q+1)^2` isotropic lines. Thus the two calculations examine 7,056 and 10,816 isotropic lines. Every one must acquire a violating scalar multiple.

In [ ]:
for primary in obstruction.primary_checks:
    prime = int(primary.prime)
    expected_projective = prime**3 + prime**2 + prime + 1
    expected_isotropic = (prime + 1)**2

    print(f"q={prime}")
    print("  component indices:", primary.component_indices)
    print("  eligible:", primary.eligible)
    print("  all projective lines:",
          primary.projective_vectors_in_search_space)
    print("  expected projective count:", expected_projective)
    print("  isotropic lines examined:",
          primary.isotropic_lines_examined)
    print("  expected isotropic count:", expected_isotropic)
    print("  lines with a violation:", primary.violating_lines)
    print("  certified:", primary.certified)

    assert primary.projective_vectors_in_search_space == expected_projective
    assert primary.isotropic_lines_examined == expected_isotropic
    assert primary.violating_lines == expected_isotropic
    assert primary.certified

The paper estimates roughly 1.6 million nonzero primary isotropic vectors before quotienting by nonzero scalar multiplication. The modern search uses canonical projective representatives, reducing that to 17,872 lines without changing the logical statement: all nonzero multiples of a line are considered through its multiplier search.

## 9. Inspect and independently verify retained witnesses

Each successful primary search retains its first violation. The optimized loop computes scaled table sums, so we now recompute each stored witness through the independent public `casson_gordon` path and inspect every signed summand contribution.

In [ ]:
for primary in obstruction.primary_checks:
    witness = primary.sample_witness
    recomputed = paper_knot.casson_gordon(
        witness.character_parameters
    )

    print(f"q={primary.prime} sample witness")
    print(asdict(witness))
    print("  independently recomputed sigma:", recomputed.sigma)
    print("  independently recomputed eta:", recomputed.eta)
    print("  signed summand sigmas:",
          tuple(summand.sigma for summand in recomputed.summands))
    print("  inequality:",
          witness.left_hand_side, ">", witness.bound)

    assert recomputed.sigma == witness.sigma
    assert recomputed.eta == witness.eta
    assert witness.left_hand_side == abs(witness.sigma)
    assert witness.left_hand_side > witness.bound

## 10. Inspect the exhaustive log

When enabled, the text file begins with the notebook's signature check and the explicit verified metabolizer basis from section 4. The core audit then records the exact knot, linking form, tested genus, and an explanation of how the isotropic-line search treats an unknown hypothetical metabolizer. Each primary section contains:

- all component Casson--Gordon lookup-table entries;
- the total projective search-space count;
- every canonical isotropic line and its exact self-pairing;
- every multiplier actually tested, its full eight-component character vector, componentwise scaled signatures, total `sigma`, `eta`, left-hand side, bound, and Boolean violation result; and
- line, primary, and global conclusions.

The algorithm stops at the first violating multiplier on each line, exactly as Algorithm 1 does. Therefore “all computations” means every computation actually performed by that optimized algorithm, not deliberately redundant work after a witness is already known.

In [ ]:
if WRITE_FULL_LOG:
    print("log path:", FULL_LOG_PATH)
    print("log size in bytes:", FULL_LOG_PATH.stat().st_size)
    print("first 20 log lines:")
    with FULL_LOG_PATH.open(encoding="utf-8") as log_file:
        for _, line in zip(range(20), log_file):
            print(line.rstrip())
else:
    print("No log was written during this clean notebook run.")
    print("Set WRITE_FULL_LOG=True and rerun sections 6--10 to create it.")

## 11. Why this calculation stops at the lower bound two

The search proves that a genus-one surface cannot exist. If one tests genus two, each primary block has four generators and can fit entirely into the rank-`2g=4` part allowed by Gilmer's theorem. The primary rank criterion is then ineligible, so this particular sufficient obstruction correctly makes no claim that the genus exceeds two.

In [ ]:
genus_two_test = paper_knot.gilmer_genus_obstruction(2)

print("certifies genus greater than two:", genus_two_test.certified)
print("reported lower bound:", genus_two_test.lower_bound)
print("primary eligibility:",
      tuple(check.eligible for check in genus_two_test.primary_checks))

This inconclusive genus-two search is expected. The exact equality in Theorem 1.1 combines three logically different inputs:

1. algebraic sliceness and nonsliceness, cited in the paper from earlier work;
2. the computer-assisted Casson--Gordon calculation above, giving `g_4^top(K) >= 2`; and
3. the explicit genus-two surface of Lemma 4.1, giving `g_4(K) <= 2`.

Together with `g_4^top(K) <= g_4(K)`, they imply `g_4^top(K)=g_4(K)=2`. The notebook reproduces item 2 completely and checks compatible algebraic data, but it does not encode the geometric band moves used for item 3.

## 12. Reference and exercises

Primary reference:

- Maria Marchwicka and Wojciech Politarczyk, *On the slice genus of generalized algebraic knots*, Journal of Knot Theory and Its Ramifications **32** (2023), article 2350085; [arXiv:2107.11299](https://arxiv.org/abs/2107.11299). See especially Theorem 1.1, Theorem 2.9, Lemma 2.12, Lemma 3.1, Algorithm 1, Lemma 4.1, and Remark 4.2.

Exercises:

1. Turn on the exhaustive log, locate the first 83-primary isotropic line, and independently add its listed component signature contributions.
2. Compare the two explicit example metabolizer bases obtained by pairing coordinates `(1,2),(3,4)` and `(1,4),(2,3)` inside one primary block.
3. For a logged isotropic line, multiply its canonical vector by a nonzero scalar and verify that the set of signature values is merely permuted.
4. Run the genus-zero obstruction and explain why its conclusion is weaker than the published genus-one calculation.